In [1]:
# --- Colab bootstrap -------------------------------------------------------
# No-op when you already have the thermo package alongside this notebook (the
# normal case: you cloned the repository and are running from code/chNN/).
# In Colab there is no repository, so fetch the package and the property data.
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    # rm -rf first: git clone REFUSES an existing directory, and under `!` that
    # failure is silent -- a half-finished earlier clone would otherwise leave an
    # empty thermo/ and surface as a baffling ImportError further down.
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    # Section IV checks every number in Table 6.4-4 against the CSV that
    # PR_properties_table_O2_example.ipynb writes, so bring that along too.
    !mkdir -p output && cp /tmp/cbet6e/code/ch6/output/*.csv output/
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# A fundamental equation of state &mdash; every property of oxygen from one function

Section 6.2 makes a claim and then proves it with algebra. Illustration 6.2-4 asks you to
*"show that from an equation of state relating the Gibbs energy, temperature, and pressure,
equations of state for all other state functions (and their derivatives as well) can be
obtained by appropriate differentiation,"* and it does exactly that for
$\underline{G}(T,P)$: entropy, volume, enthalpy, internal energy, the Helmholtz energy,
both heat capacities, $\kappa_T$ and $\alpha$ all fall out as derivatives.

Section 6.4 then points out that modern equations of state are written this way, but in the
*other* pair of variables &mdash; the Helmholtz energy as a function of temperature and
density:

$$\underline{A} = \underline{A}(T, \underline{V})$$

and it names the 1995 IAPWS formulation for water, with its 56 parameters, as an example.
What the chapter never does is **show one working**. That is this notebook.

### What is demonstrated here

We build $\underline{A}(T,\underline{V})$ for oxygen out of the chapter's own data &mdash;
nothing else &mdash; and then obtain

$$P,\quad Z,\quad \underline{S},\quad \underline{U},\quad \underline{H},\quad
\underline{G},\quad C_V,\quad C_P,\quad \phi,\quad \mu$$

**by differentiating that one function numerically.** No property formula is quoted
anywhere below: not Eq. 6.4-2 for the pressure, not Eq. 6.4-29 or 6.4-30 for the
departures. They are then used as the *check*, which is the point &mdash; if the
fundamental equation really contains all the information about the fluid, the chapter's
individually derived equations must come back out of it, and every number in Table 6.4-4
must too.

### Why this matters beyond the arithmetic

A fundamental equation is not a convenience. It is a statement about what a fluid *is*:
one scalar function of two variables, from which everything measurable follows. That is
why reference-quality property data &mdash; NIST's tables, IAPWS-95 for water, the
Schmidt&ndash;Wagner equation behind NIST's oxygen tables &mdash; is published in this form
rather than as a volumetric equation plus a pile of separate correlations. Fit one
function well and you have fitted the fluid.

> **Section 6.2's caution has aged.** After Illustration 6.2-4 the text says it is
> *"unlikely that such equations will be available for all fluids of interest to
> engineers."* That was fair when it was written. Reference-quality fundamental equations
> now exist for on the order of a hundred and fifty fluids and are distributed in software
> libraries. The scarce thing today is not the equation; it is knowing what it is and what
> it can be asked for.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst  
August 2026

In [2]:
import sys
sys.path.append("..")           # so `import thermo` finds code/thermo

import numpy as np
import pandas as pd
from scipy.integrate import quad

from thermo import PengRobinson, APPENDIX_A2_CP_CRYO, TABLE_6_6_1
from thermo.peng_robinson import R

SQ2 = np.sqrt(2.0)
T_REF, P_REF = 298.15, 1e5       # the reference state of Illustration 6.4-1:
                                 # the IDEAL GAS at 25 C and 1 bar

# Oxygen, from the book's own printed data and nothing else: Table 6.6-1 for the
# critical constants and the acentric factor, Appendix A.II (cryogenic row) for the
# ideal-gas heat capacity. Exactly the inputs Illustration 6.4-1 uses.
pr = PengRobinson(**TABLE_6_6_1["oxygen"], cp=APPENDIX_A2_CP_CRYO["oxygen"])
cA, cB, cC, cD = pr.cp

print(f"oxygen:  Tc = {pr.Tc} K   Pc = {pr.Pc/1e6:.3f} MPa   omega = {pr.omega}")
print(f"         kappa = {pr.kappa:.6f}   b = {pr.b:.6e} m3/mol")
print(f"Cp* = {cA} + {cB:g} T + {cC:g} T^2 + {cD:g} T^3   J/(mol K)")
print(f"reference state: ideal gas at {T_REF-273.15:.0f} C, {P_REF/1e5:.0f} bar")

oxygen:  Tc = 154.6 K   Pc = 5.046 MPa   omega = 0.021
         kappa = 0.406908   b = 1.981874e-05 m3/mol
Cp* = 30.171 + -0.01293 T + 4.236e-05 T^2 + -2.5828e-08 T^3   J/(mol K)
reference state: ideal gas at 25 C, 1 bar


## I. The two halves of $\underline{A}$

Every fundamental equation in use is written as an **ideal-gas part plus a residual
part** &mdash; that is the structure of IAPWS-95 and of every equation like it:

$$\underline{A}(T,\underline{V}) \;=\;
\underbrace{\underline{A}^{\mathrm{IG}}(T,\underline{V})}_{\text{one molecule at a time}}
\;+\;
\underbrace{\underline{A}^{\mathrm{res}}(T,\underline{V})}_{\text{everything intermolecular}}$$

**The ideal-gas part needs only $C_P^{*}$.** With $\underline{H}^{\mathrm{IG}}$ and
$\underline{S}^{\mathrm{IG}}$ built from the heat capacity in the usual way,

$$\underline{A}^{\mathrm{IG}} = \underline{U}^{\mathrm{IG}} - T\underline{S}^{\mathrm{IG}},
\qquad \underline{U}^{\mathrm{IG}}(T) = \underline{H}^{\mathrm{IG}}(T) - RT,
\qquad \underline{S}^{\mathrm{IG}}(T,\underline{V})
= \int_{T_{\mathrm{ref}}}^{T}\frac{C_P^{*}}{T'}\,\mathrm{d}T'
- R\ln\frac{RT/\underline{V}}{P_{\mathrm{ref}}}$$

**The $-RT$ in $\underline{U}^{\mathrm{IG}}$ is not decoration, and getting it wrong is
the single easiest mistake here.** The reference state fixes $\underline{H}$ and
$\underline{S}$, not $\underline{U}$; if you instead force $\underline{U}^{\mathrm{IG}} = 0$
at the reference temperature, every enthalpy this notebook produces comes out high by
exactly $RT_{\mathrm{ref}} = 2479$ J/mol. It is a constant offset, so it survives every
internal consistency check and only shows up when you compare with the book.

**The residual part is one integration of the volumetric equation of state.** This is the
only place the Peng&ndash;Robinson equation enters:

$$\underline{A}^{\mathrm{res}}(T,\underline{V})
= \int_{\underline{V}}^{\infty}\left(P - \frac{RT}{\underline{V}'}\right)\mathrm{d}\underline{V}'$$

The integrand is the difference between the real pressure and the ideal-gas pressure at the
same $T$, so it vanishes as $\underline{V}'\to\infty$ and the integral converges. Done
analytically for Peng&ndash;Robinson it gives

$$\underline{A}^{\mathrm{res}} = -RT\ln\!\left(1 - \frac{b}{\underline{V}}\right)
- \frac{a(T)}{2\sqrt{2}\,b}
\ln\!\left[\frac{\underline{V}+(1+\sqrt2)b}{\underline{V}+(1-\sqrt2)b}\right]$$

and the cell below **checks that closed form against the integral evaluated numerically**,
so the algebra is not taken on trust. Any other volumetric equation of state would go
through the same integral and produce a different $\underline{A}^{\mathrm{res}}$;
everything after Section I is untouched by that choice.

In [3]:
def H_ig(T):
    """Ideal-gas molar enthalpy relative to T_REF, J/mol."""
    return (cA*(T - T_REF) + cB/2*(T**2 - T_REF**2)
            + cC/3*(T**3 - T_REF**3) + cD/4*(T**4 - T_REF**4))


def S_ig_T(T):
    """The temperature part of the ideal-gas entropy, J/(mol K)."""
    return (cA*np.log(T/T_REF) + cB*(T - T_REF)
            + cC/2*(T**2 - T_REF**2) + cD/3*(T**3 - T_REF**3))


def A_ig(T, V):
    """Ideal-gas Helmholtz energy at (T, V), J/mol."""
    S = S_ig_T(T) - R*np.log((R*T/V)/P_REF)
    U = H_ig(T) - R*T                    # so that U + RT == H_ig(T) exactly
    return U - T*S


def A_res(T, V):
    """Residual Helmholtz energy at (T, V), J/mol -- the PR integral, closed form."""
    b = pr.b
    log_term = np.log((V + (1+SQ2)*b)/(V + (1-SQ2)*b))
    return -R*T*np.log1p(-b/V) - pr.a(T)/(2*SQ2*b)*log_term


def A(T, V):
    """THE FUNDAMENTAL EQUATION. Everything below is a derivative of this."""
    return A_ig(T, V) + A_res(T, V)


# Check the closed form against the integral it came from, at three states.
print(f"{'T (K)':>8}{'V (m3/mol)':>14}{'A_res closed':>15}{'A_res quad':>13}{'diff':>11}")
for T, V in ((173.15, 1.4e-4), (273.15, 4.3e-4), (423.15, 3.4e-4)):
    num, err = quad(lambda v: pr.pressure(v, T) - R*T/v, V, np.inf, limit=200)
    print(f"{T:8.2f}{V:14.3e}{A_res(T,V):15.6f}{num:13.6f}{A_res(T,V)-num:11.2e}")
    assert abs(A_res(T, V) - num) < 1e-6*abs(num), "closed form != the integral"

print("\nclosed form and numerical integration agree -- the algebra is verified,")
print("not assumed.")

   T (K)    V (m3/mol)   A_res closed   A_res quad       diff
  173.15     1.400e-04    -682.481181  -682.481181   4.09e-12
  273.15     4.300e-04    -142.804048  -142.804048   1.99e-12
  423.15     3.400e-04     -13.158045   -13.158045  -2.13e-13

closed form and numerical integration agree -- the algebra is verified,
not assumed.


## II. Every property is a derivative of $\underline{A}$

This is the whole content of a fundamental equation, and it is worth seeing the list in one
place. $\underline{A}$ is a thermodynamic potential in $(T,\underline{V})$, so

$$\mathrm{d}\underline{A} = -\underline{S}\,\mathrm{d}T - P\,\mathrm{d}\underline{V}$$

and therefore

| property | from $\underline{A}$ |
|---|---|
| pressure | $P = -\left(\partial \underline{A}/\partial \underline{V}\right)_T$ |
| entropy | $\underline{S} = -\left(\partial \underline{A}/\partial T\right)_{\underline{V}}$ |
| internal energy | $\underline{U} = \underline{A} + T\underline{S}$ |
| enthalpy | $\underline{H} = \underline{U} + P\underline{V}$ |
| Gibbs energy | $\underline{G} = \underline{A} + P\underline{V}$ |
| constant-volume heat capacity | $C_V = -T\left(\partial^2 \underline{A}/\partial T^2\right)_{\underline{V}}$ |
| constant-pressure heat capacity | $C_P = C_V - T\dfrac{\left(\partial P/\partial T\right)_{\underline{V}}^{2}}{\left(\partial P/\partial \underline{V}\right)_T}$ |
| fugacity coefficient | $\ln\phi = \dfrac{\underline{A}^{\mathrm{res}}}{RT} + Z - 1 - \ln Z$ |
| Joule&ndash;Thomson coefficient | $\mu = -\dfrac{\underline{V} - T(\partial \underline{V}/\partial T)_P}{C_P}$ |

The last three rows are still only derivatives of $\underline{A}$: $C_P$ needs the two
second derivatives that make up $(\partial P/\partial T)_{\underline{V}}$ and
$(\partial P/\partial \underline{V})_T$, and $\mu$ needs
$(\partial \underline{V}/\partial T)_P = -(\partial P/\partial T)_{\underline{V}} /
(\partial P/\partial \underline{V})_T$ &mdash; the triple-product rule of Sec. 6.2.

**Everything is computed by central differences below, on purpose.** Differentiating
$\underline{A}$ by hand would produce the chapter's equations, which is the thing being
tested; doing it numerically means no algebra is hidden, and the same six lines work for
*any* $\underline{A}$, including a 56-term fitted one.

*A word on step size, since second derivatives are involved.* A central second difference
divides by $h^2$, so too small an $h$ loses the answer in round-off and too large an $h$
biases it. The steps below are **relative** &mdash; $h_T = 10^{-3}T$ and
$h_V = 10^{-5}\underline{V}$ &mdash; which keeps $\sim$8 significant figures in $C_V$
across the whole table. Section III checks $C_P$ against a completely different route as
evidence that the choice is sound.

In [4]:
def properties(T, V, hT=None, hV=None):
    """Every property at (T, V), by central differences on A. Nothing is quoted."""
    hT = hT if hT is not None else 1e-3*T
    hV = hV if hV is not None else 1e-5*V

    A0 = A(T, V)
    A_T = (A(T+hT, V) - A(T-hT, V))/(2*hT)
    A_V = (A(T, V+hV) - A(T, V-hV))/(2*hV)
    A_TT = (A(T+hT, V) - 2*A0 + A(T-hT, V))/hT**2
    A_VV = (A(T, V+hV) - 2*A0 + A(T, V-hV))/hV**2
    A_TV = (A(T+hT, V+hV) - A(T+hT, V-hV)
            - A(T-hT, V+hV) + A(T-hT, V-hV))/(4*hT*hV)

    P = -A_V
    S = -A_T
    U = A0 + T*S
    Z = P*V/(R*T)
    Cv = -T*A_TT
    dPdT, dPdV = -A_TV, -A_VV
    dVdT_P = -dPdT/dPdV                       # triple-product rule
    Cp = Cv - T*dPdT**2/dPdV

    return dict(A=A0, P=P, S=S, U=U, H=U + P*V, G=A0 + P*V, Z=Z,
                Cv=Cv, Cp=Cp, dPdT=dPdT, dPdV=dPdV, dVdT_P=dVdT_P,
                alpha=dVdT_P/V, kappa_T=-1/(V*dPdV),
                mu=-(V - T*dVdT_P)/Cp,
                ln_phi=A_res(T, V)/(R*T) + Z - 1 - np.log(Z))


T, P = 273.15, 50e5                           # 0 C, 50 bar -- a cell of Table 6.4-4
V = max(pr.physical_Z(T, P))*R*T/P
p = properties(T, V)

print(f"oxygen at {T-273.15:.0f} C and {P/1e5:.0f} bar,  V = {V:.4e} m3/mol\n")
for k, unit in (("A", "J/mol"), ("U", "J/mol"), ("H", "J/mol"), ("G", "J/mol"),
                ("S", "J/(mol K)"), ("Cv", "J/(mol K)"), ("Cp", "J/(mol K)"),
                ("Z", "-"), ("ln_phi", "-")):
    print(f"  {k:>7} = {p[k]:14.6f}  {unit}")
print(f"  {'alpha':>7} = {p['alpha']:14.6e}  1/K       (Eq. 6.2-3)")
print(f"  {'kappa_T':>7} = {p['kappa_T']:14.6e}  1/Pa      (Eq. 6.2-4)")
print(f"  {'mu':>7} = {p['mu']*1e5:14.6f}  K/bar     (Eq. 6.2-27)")
print("\nAll of it from one function and five finite differences.")

oxygen at 0 C and 50 bar,  V = 4.2816e-04 m3/mol

        A =    6572.638081  J/mol
        U =   -3412.174904  J/mol
        H =   -1271.373737  J/mol
        G =    8713.439248  J/mol
        S =     -36.554322  J/(mol K)
       Cv =      21.425002  J/(mol K)
       Cp =      32.952057  J/(mol K)
        Z =       0.942629  -
   ln_phi =      -0.061413  -
    alpha =   4.553384e-03  1/K       (Eq. 6.2-3)
  kappa_T =   2.103574e-07  1/Pa      (Eq. 6.2-4)
       mu =       0.316724  K/bar     (Eq. 6.2-27)

All of it from one function and five finite differences.


## III. Does the chapter's own arithmetic come back out?

Now the test. Each of these was derived separately in the text, by a different route, and
none of them was used above:

| what | the chapter's equation |
|---|---|
| pressure | Eq. 6.4-2, the Peng&ndash;Robinson equation itself |
| enthalpy departure | Eq. 6.4-29 |
| entropy departure | Eq. 6.4-30 |
| fugacity coefficient | the $\ln\phi$ expression of Sec. 7.4 |
| $C_P - C_V$ | Eq. 6.2-35, $\;T\underline{V}\alpha^2/\kappa_T$ |

If the fundamental equation contains all the information about the fluid, all five must
agree with what came out of $\underline{A}$ &mdash; to finite-difference precision, not to
the last bit.

$C_P$ gets a second, independent check: differentiate $\underline{H}(T,P)$ with respect to
$T$ at **constant pressure**, which goes nowhere near the second derivatives of
$\underline{A}$ and so would not share their error.

In [5]:
def S_book(T, P):
    """S(T,P) the chapter's way: ideal gas at (T,P) plus the Eq. 6.4-30 departure."""
    return S_ig_T(T) - R*np.log(P/P_REF) + pr.departure_S(T, P)


def H_book(T, P):
    """H(T,P) the chapter's way: ideal-gas integral plus the Eq. 6.4-29 departure."""
    return H_ig(T) + pr.departure_H(T, P)


rows = []
rows.append(("P", "Eq. 6.4-2", p["P"], pr.pressure(V, T)))
rows.append(("H", "Eq. 6.4-29", p["H"], H_book(T, P)))
rows.append(("S", "Eq. 6.4-30", p["S"], S_book(T, P)))
rows.append(("ln phi", "Sec. 7.4", p["ln_phi"], pr.ln_phi(T, P)))
rows.append(("Cp - Cv", "Eq. 6.2-35", p["Cp"] - p["Cv"],
             T*V*p["alpha"]**2/p["kappa_T"]))

# an independent Cp: dH/dT at constant P, through the EOS root each time
def H_of_TP(Tx, Px):
    Vx = max(pr.physical_Z(Tx, Px))*R*Tx/Px
    return properties(Tx, Vx)["H"]

h = 0.05
rows.append(("Cp", "dH/dT at fixed P", p["Cp"],
             (H_of_TP(T+h, P) - H_of_TP(T-h, P))/(2*h)))

hdr = f"{'property':<10}{'from A':>18}{'from the book':>18}{'difference':>14}{'checked against':>22}"
print(hdr); print("-"*len(hdr))
for name, src, mine, theirs in rows:
    print(f"{name:<10}{mine:18.8f}{theirs:18.8f}{mine-theirs:14.2e}{src:>22}")
    scale = max(abs(theirs), 1.0)
    assert abs(mine - theirs) < 1e-4*scale, f"{name} disagrees with {src}"

print("\nEvery one agrees. The chapter derived these five results independently;")
print("they are all consequences of the single function A(T,V).")

property              from A     from the book    difference       checked against
----------------------------------------------------------------------------------
P           5000000.00002301  5000000.00000000      2.30e-05             Eq. 6.4-2
H             -1271.37373716    -1271.37280628     -9.31e-04            Eq. 6.4-29
S               -36.55432175      -36.55431834     -3.41e-06            Eq. 6.4-30
ln phi           -0.06141301       -0.06141301     -2.64e-13              Sec. 7.4
Cp - Cv          11.52705531       11.52705531      3.55e-15            Eq. 6.2-35
Cp               32.95205710       32.95240822     -3.51e-04      dH/dT at fixed P

Every one agrees. The chapter derived these five results independently;
they are all consequences of the single function A(T,V).


## IV. All of Table 6.4-4, from one function

The strongest form of the test. Table 6.4-4 is 143 states $\times$ four properties = 572
printed numbers, generated by `PR_properties_table_O2_example.ipynb` from the departure
functions. Here every one of them is recomputed as a **derivative of
$\underline{A}(T,\underline{V})$**, and compared against that notebook's own CSV output.

The table prints $\underline{H}$ and $\underline{S}$ to two decimals, so the bar to clear
is 0.01 &mdash; agreement well inside that means a reader could have built the entire table
from the fundamental equation and got the same book.

In [6]:
csv = "output/Table_6.4-4.csv"
try:
    tbl = pd.read_csv(csv)
except FileNotFoundError:
    raise SystemExit(
        f"{csv} not found -- run PR_properties_table_O2_example.ipynb first, "
        "or re-run the bootstrap cell at the top to fetch it")

worst = {"Z": 0.0, "V": 0.0, "H": 0.0, "S": 0.0}
where = {}
for _, r in tbl.iterrows():
    Tk, Pa = r.T_C + 273.15, r.P_bar*1e5
    Vm = max(pr.physical_Z(Tk, Pa))*R*Tk/Pa
    q = properties(Tk, Vm)
    for key, mine, theirs in (("Z", q["Z"], r.Z),
                              ("V", Vm*1e3, r.V_m3_per_kmol),
                              ("H", q["H"], r.H_J_per_mol),
                              ("S", q["S"], r.S_J_per_mol_K)):
        d = abs(mine - theirs)
        if d > worst[key]:
            worst[key], where[key] = d, (r.P_bar, r.T_C)

print(f"{len(tbl)} states x 4 properties = {4*len(tbl)} numbers, "
      f"every one a derivative of A\n")
print(f"{'column':<8}{'worst |difference|':>22}{'at':>18}{'printed to':>14}")
print("-"*62)
for key, unit, printed in (("Z", "", "0.0001"), ("V", " m3/kmol", "0.0001"),
                           ("H", " J/mol", "0.01"), ("S", " J/(mol K)", "0.01")):
    pb, tc = where[key]
    print(f"{key:<8}{worst[key]:>16.2e}{unit:<6}{f'{pb:g} bar, {tc:g} C':>18}{printed:>14}")

assert worst["H"] < 1e-2 and worst["S"] < 1e-2, "a column misses the printed precision"
assert worst["Z"] < 1e-6 and worst["V"] < 1e-6
print("\nEvery column agrees to far better than the precision the book prints.")
print("Table 6.4-4 is a property of one scalar function of two variables.")

143 states x 4 properties = 572 numbers, every one a derivative of A

column      worst |difference|                at    printed to
--------------------------------------------------------------
Z               1.10e-10           30 bar, -50 C        0.0001
V               3.55e-15 m3/kmol       1 bar, 50 C        0.0001
H               1.33e-03 J/mol    100 bar, 150 C          0.01
S               4.31e-06 J/(mol K)   100 bar, -100 C          0.01

Every column agrees to far better than the precision the book prints.
Table 6.4-4 is a property of one scalar function of two variables.


## V. What a fitted residual buys, and what it costs

Nothing above depends on $\underline{A}^{\mathrm{res}}$ having come from
Peng&ndash;Robinson. Replace that one function and every derivative in Section II still
holds. That is precisely what a reference-quality equation does:

$$\frac{\underline{A}^{\mathrm{res}}(\tau,\delta)}{RT}
= \sum_{i} n_i\,\delta^{d_i}\tau^{t_i}
\;+\; \sum_{j} n_j\,\delta^{d_j}\tau^{t_j}\mathrm{e}^{-\gamma_j\delta^{p_j}}
\;+\;\cdots
\qquad \tau = \frac{T_c}{T},\quad \delta = \frac{\rho}{\rho_c}$$

with the coefficients fitted simultaneously to $PVT$ data, heat capacities, speeds of
sound, and vapor pressures. Peng&ndash;Robinson has **two** adjustable quantities,
$a(T)$ and $b$, generalized from $T_c$, $P_c$ and $\omega$. For comparison:

| equation | fluid | terms | typical density accuracy |
|---|---|---|---|
| Peng&ndash;Robinson | any, from $T_c,P_c,\omega$ | 2 | a few percent |
| Schmidt&ndash;Wagner (1985) | oxygen | tens | ~0.1% or better |
| IAPWS-95 | water | 56 | ~0.01% or better |

Illustration 6.4-1's *Comments* names the first of those: for oxygen, the reference
equation is the fundamental equation of Schmidt and Wagner, *"explicit in the Helmholtz
energy in the manner described earlier in this section"* &mdash; the manner this notebook
has just carried out &mdash; *"and it is the source of the oxygen properties tabulated by
the National Institute of Standards and Technology."* So the NIST oxygen tables and this
notebook are the **same calculation with a better $\underline{A}^{\mathrm{res}}$**.

**The coefficients are deliberately not transcribed here.** A fitted table of tens of
numbers copied from memory or from a secondary source is worthless &mdash; worse than
worthless, because it looks authoritative. Getting them right means taking them from the
primary source and then proving the implementation reproduces that source's own published
verification values. Both of the equations above ship such a table for exactly this
purpose. That is the exercise below, and it is the real skill: **not differentiating
$\underline{A}$, which you have now seen is six lines, but establishing that the
$\underline{A}$ you typed in is the one that was published.**

### What Peng&ndash;Robinson costs, in this notebook's own terms

Worth stating plainly, because Section IV's agreement can be misread. Sections III and IV
show that this notebook and Table 6.4-4 are *the same calculation* &mdash; they agree to
$10^{-3}$ J/mol. They do **not** show that either one is accurate. Both rest on the same
two-parameter $\underline{A}^{\mathrm{res}}$, and both inherit the same ideal-gas
$C_P^{*}$. Consistency and accuracy are different claims, and only the first has been
established here.

## Exercises

**1. The reference constant.** Change $\underline{U}^{\mathrm{IG}}(T)$ in Section I from
$\underline{H}^{\mathrm{IG}}(T) - RT$ to $\underline{H}^{\mathrm{IG}}(T) - R(T -
T_{\mathrm{ref}})$ &mdash; the choice that makes $\underline{U}^{\mathrm{IG}} = 0$ at the
reference temperature. Which of Sections III and IV still pass, and which fail? Explain why
the internal checks are blind to it, and what that says about the difference between a
consistency check and a correctness check.

**2. A different volumetric equation.** Derive $\underline{A}^{\mathrm{res}}$ for the van
der Waals equation by evaluating the Section I integral, implement it, and verify it against
`quad` the way `A_res` is verified above. Then rerun Sections II and III unchanged. How
large is the disagreement with Table 6.4-4, and at which states is it worst? *(Everything
except `A_res` should require no edits at all. If it does, something has leaked.)*

**3. Where the second derivatives break down.** Sweep the step sizes $h_T$ and $h_V$ over
several decades and plot the error in $C_V$ against each. Identify the round-off-dominated
and truncation-dominated branches and the optimum between them. Why is $C_V$ so much more
delicate than $P$?

**4. The critical point, from $\underline{A}$ alone.** The critical point satisfies
$(\partial P/\partial \underline{V})_T = (\partial^2 P/\partial \underline{V}^2)_T = 0$
(Eq. 6.6-1) &mdash; that is, the *second and third* derivatives of $\underline{A}$ with
respect to $\underline{V}$ both vanish. Solve those two equations numerically for $T$ and
$\underline{V}$, and confirm you recover $T_c = 154.6$ K and $P_c = 5.046$ MPa. What
happens to $C_P$ as you approach that state, and why?

**5. A reference-quality residual.** Obtain the coefficients of a published
Helmholtz-explicit fundamental equation for a pure fluid &mdash; IAPWS-95 for water is the
best documented &mdash; **from the primary source**. Implement its
$\underline{A}^{\mathrm{res}}(\tau,\delta)$, and *before computing anything else*, verify
your implementation against the verification values the source publishes for that purpose.
Only then reuse Section II's `properties()` unchanged. Compare its $\underline{H}$ and
$\underline{S}$ with the Peng&ndash;Robinson values along one isotherm, and say where a
two-parameter equation is good enough and where it is not.

*Exercise 5 is the one worth the time. Exercises 1 through 4 rehearse the machinery; 5 is
the thing practicing engineers actually do with it.*